# 08 — Sparkov Final Test Evaluation (M3 LightGBM)

**Purpose:** select a daily top-k alert policy on saved M3 validation predictions, then evaluate the locked M3 model on the untouched chronological test period. Adapted from IEEE-CIS Notebook 10; M3 training logic copied from Sparkov Notebook 6.

**Leakage controls:** model parameters and k are fixed before test evaluation; at each weekly cutoff, only labels older than seven days are verified; transactions from the most recent seven days may be pseudo-labelled without consulting their true labels. Later test labels can become available for subsequent retraining only after the delay. Do not use test outcomes to reselect the model or k.

**Execution:** run cells top-to-bottom from the `notebooks/` directory. The test run is computationally expensive; results are cached to CSV.

In [1]:
from pathlib import Path
from itertools import product
import time
import json
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/sparkov_final_test")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load the processed feature table and the pre-approved modelling feature list.
df = pd.read_pickle(DATA_DIR / "sparkov_features.pkl")
with open(DATA_DIR / "feature_lists.json", "r") as f:
    feature_lists = json.load(f)
feature_cols = feature_lists["sparkov_feature_cols"]

TIME_COL = "trans_date_trans_time"
TARGET_COL = "is_fraud"
SPLIT_COL = "sparkov_split"

# Final evaluation: include the untouched test partition without using its labels for model or k selection.
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
m3_df = df.loc[df[SPLIT_COL].isin(["train", "validation", "test"])].copy()
m3_df = m3_df.sort_values(TIME_COL, kind="stable").reset_index(drop=True)
del df

# Match M2: reconstruct selected frequency features using strictly earlier rows.
frequency_sources = {
    "merchant_freq": "merchant",
    "category_freq": "category",
    "state_freq": "state",
}
for feature, source in frequency_sources.items():
    if feature not in feature_cols:
        continue
    previous_rows = m3_df.groupby(source, dropna=False).cumcount()
    same_timestamp_previous = m3_df.groupby(
        [source, TIME_COL], dropna=False
    ).cumcount()
    m3_df[feature] = (previous_rows - same_timestamp_previous).astype("int64")

train_mask = m3_df[SPLIT_COL].eq("train")
X_train_schema = m3_df.loc[train_mask, feature_cols]
categorical_cols = X_train_schema.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Data shape:", m3_df.shape)
print("Features:", len(feature_cols))
print("Categorical features:", categorical_cols)
print("Fraud rates by split:")
display(m3_df.groupby(SPLIT_COL)[TARGET_COL].agg(["size", "mean"]))


Data shape: (1852394, 87)
Features: 62
Categorical features: ['category', 'gender', 'state']
Fraud rates by split:


,size,mean
sparkov_split,,
test,555719,0.003860
train,1037340,0.005753
validation,259335,0.005931


In [2]:
# Fixed M2 temporal configuration
WINDOW_DAYS = None                 # expanding history
RETRAIN_EVERY_DAYS = 7
LABEL_DELAY_DAYS = 7
# Compact, prespecified 4-parameter grid (16 configurations).
# IEEE's selection rule and class-specific reliability weights are retained.
FRAUD_THRESHOLDS = [0.15, 0.50]
LEGIT_THRESHOLDS = [0.25, 0.50]
FRAUD_RELIABILITIES = [0.15, 0.50]
LEGIT_RELIABILITIES = [0.30, 0.75]


def make_lightgbm(class_weight="balanced"):
    """Best LightGBM tree settings from M1/M2."""
    return lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        objective="binary",
        class_weight=class_weight,
        is_unbalance=False,
        random_state=100,
        n_jobs=2,
        verbosity=-1,
    )


def prepare_features(frame, categories=None):
    X = frame[feature_cols].copy()
    for col in categorical_cols:
        if categories is None:
            X[col] = X[col].astype("category")
        else:
            X[col] = pd.Categorical(X[col], categories=categories[col])
    return X


def verified_training_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select verified labels available at this cutoff; use expanding history."""
    cutoff = pd.Timestamp(cutoff)
    latest_eligible = cutoff - pd.Timedelta(days=label_delay_days)
    mask = frame[SPLIT_COL].isin(allowed_splits) & frame[TIME_COL].lt(latest_eligible)
    return frame.loc[mask].copy()


def pseudo_candidate_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select rows in the seven-day label-delay interval before the cutoff."""
    cutoff = pd.Timestamp(cutoff)
    start = cutoff - pd.Timedelta(days=label_delay_days)
    mask = (
        frame[SPLIT_COL].isin(allowed_splits)
        & frame[TIME_COL].ge(start)
        & frame[TIME_COL].lt(cutoff)
    )
    return frame.loc[mask].copy()


def make_cutoffs(start, end, frequency_days=RETRAIN_EVERY_DAYS):
    return pd.date_range(
        start=pd.Timestamp(start),
        end=pd.Timestamp(end),
        freq=f"{frequency_days}D",
        inclusive="left",
    )


def precompute_pseudo_scores(
    frame,
    start,
    end,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Cache one verified-only model's scores for each cutoff's delay window."""
    cache = {}
    cutoffs = make_cutoffs(start, end, frequency_days)

    for i, cutoff in enumerate(cutoffs, start=1):
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        candidates = pseudo_candidate_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        if verified.empty or verified[TARGET_COL].nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")
        if candidates.empty:
            cache[cutoff] = pd.DataFrame(
                columns=["row_index", "fraud_probability"]
            )
            continue

        X_verified = prepare_features(verified)
        categories = {col: X_verified[col].cat.categories for col in categorical_cols}
        X_candidates = prepare_features(candidates, categories=categories)
        model = make_lightgbm(class_weight="balanced")
        model.fit(X_verified, verified[TARGET_COL].astype(int))
        fraud_probability = model.predict_proba(X_candidates)[:, 1]

        cache[cutoff] = pd.DataFrame({
            "row_index": candidates.index.to_numpy(),
            "fraud_probability": fraud_probability,
        })
        print(
            f"Cached {i}/{len(cutoffs)} cutoffs: {cutoff} "
            f"({len(verified):,} verified; {len(candidates):,} unlabeled candidates)",
            flush=True,
        )
    return cache


def balanced_class_weight_map(y_verified):
    """Balanced class weights based only on verified labels at this cutoff."""
    classes = np.sort(pd.Series(y_verified).astype(int).unique())
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=pd.Series(y_verified).astype(int).to_numpy(),
    )
    return {int(label): float(weight) for label, weight in zip(classes, weights)}


def summarize_predictions(predictions):
    y_true = predictions["y_true"].astype(int)
    y_score = predictions["y_score"].astype(float)
    return {
        "transactions": len(predictions),
        "fraud_cases": int(y_true.sum()),
        "pr_auc": average_precision_score(y_true, y_score),
        "roc_auc": roc_auc_score(y_true, y_score),
    }


In [3]:
def run_adaptive_lightgbm(
    frame,
    start,
    end,
    fraud_thres=None,
    legit_thres=None,
    fraud_reliability=0.0,
    legit_reliability=0.0,
    pseudo_score_cache=None,
    allowed_splits=("train",),
    scoring_split="train",
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Weekly LightGBM retraining; None fraud_thres gives verified-only M2."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if start >= end:
        raise ValueError("start must be earlier than end")
    use_pseudo = fraud_thres is not None
    if use_pseudo:
        if legit_thres is None:
            raise ValueError("legit_thres is required for M3")
        if not 0.0 < fraud_thres <= 1.5 or not 0.0 < legit_thres < 1.0:
            raise ValueError("fraud_thres must be in (0, 1.5], legit_thres in (0, 1)")
        if not 0 <= fraud_reliability <= 1 or not 0 <= legit_reliability <= 1:
            raise ValueError("Reliability weights must be in [0, 1]")

    prediction_blocks = []
    for cutoff in make_cutoffs(start, end, frequency_days):
        next_cutoff = min(cutoff + pd.Timedelta(days=frequency_days), end)
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        score_block = frame.loc[
            frame[SPLIT_COL].eq(scoring_split)
            & frame[TIME_COL].ge(cutoff)
            & frame[TIME_COL].lt(next_cutoff)
        ].copy()
        if score_block.empty:
            continue
        y_verified = verified[TARGET_COL].astype(int)
        if verified.empty or y_verified.nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")

        pseudo_rows = frame.iloc[0:0].copy()
        pseudo_rows["_m3_target"] = pd.Series(dtype=int)
        pseudo_rows["_m3_reliability"] = pd.Series(dtype=float)
        if use_pseudo:
            if pseudo_score_cache is None or cutoff not in pseudo_score_cache:
                raise KeyError(f"Missing pseudo-label scores for {cutoff}")
            scored_candidates = pseudo_score_cache[cutoff]
            if not scored_candidates.empty:
                # Same rank-based top/bottom selection as IEEE M3.
                p = scored_candidates["fraud_probability"].to_numpy()
                sorted_scores = np.sort(p)
                verified_rate = float(y_verified.mean())
                n_fraud = min(max(int(round(fraud_thres * verified_rate * len(p))), 1), len(p))
                n_legit = min(max(int(round(legit_thres * len(p))), 1), len(p))
                fraud_cutoff = sorted_scores[-n_fraud]
                legit_cutoff = sorted_scores[n_legit - 1]
                is_fraud = p >= fraud_cutoff
                is_legit = p <= legit_cutoff
                # IEEE's implementation gives fraud precedence when masks overlap.
                keep = is_fraud | is_legit
                chosen = scored_candidates.loc[keep]
                if not chosen.empty:
                    pseudo_rows = frame.loc[chosen["row_index"].astype(int).to_numpy()].copy()
                    pseudo_rows["_m3_target"] = is_fraud[keep].astype(int)
                    pseudo_rows["_m3_reliability"] = np.where(
                        is_fraud[keep], fraud_reliability, legit_reliability
                    )

        verified_rows = verified.copy()
        verified_rows["_m3_target"] = y_verified.to_numpy()
        fit_rows = pd.concat([verified_rows, pseudo_rows], axis=0)
        y_fit = fit_rows["_m3_target"].astype(int)

        # Per-row sample weight = class weight from verified labels × reliability.
        class_weights = balanced_class_weight_map(y_verified)
        class_component = y_fit.map(class_weights).to_numpy(dtype=float)
        reliability_component = np.concatenate([
            np.ones(len(verified_rows), dtype=float),
            pseudo_rows["_m3_reliability"].to_numpy(dtype=float),
        ])
        fit_sample_weight = class_component * reliability_component

        X_fit = prepare_features(fit_rows)
        categories = {col: X_fit[col].cat.categories for col in categorical_cols}
        X_score = prepare_features(score_block, categories=categories)
        # Class balance is already represented in fit_sample_weight, avoiding double weighting.
        model = make_lightgbm(class_weight=None)
        model.fit(X_fit, y_fit, sample_weight=fit_sample_weight)
        scores = model.predict_proba(X_score)[:, 1]

        prediction_blocks.append(pd.DataFrame({
            "row_index": score_block.index.to_numpy(),
            "timestamp": score_block[TIME_COL].to_numpy(),
            "y_true": score_block[TARGET_COL].astype(int).to_numpy(),
            "y_score": scores,
            "cutoff": cutoff,
            "fraud_thres": fraud_thres,
            "legit_thres": legit_thres,
            "fraud_reliability": fraud_reliability,
            "legit_reliability": legit_reliability,
            "verified_rows": len(verified_rows),
            "pseudo_rows": len(pseudo_rows),
            "class_weight_legitimate": class_weights[0],
            "class_weight_fraud": class_weights[1],
        }))

    if not prediction_blocks:
        raise ValueError("No scoring rows found")
    predictions = pd.concat(prediction_blocks, ignore_index=True)
    if predictions["row_index"].duplicated().any():
        raise AssertionError("Duplicate scored rows")
    return predictions


## 1. Load the locked configuration and saved validation predictions

This uses the **existing** Notebook 6 outputs; it does not repeat the 16-configuration development search or validation model training. The validation `row_index` must still identify the same transactions after loading the full processed feature table; explicit checks below prevent a silent mismatch.

In [4]:
M3_RESULTS = Path("../results/m3_rank_aligned_ieee")
config_path = M3_RESULTS / "m3_best_config.csv"
val_path = M3_RESULTS / "m3_validation_predictions.csv"
for path in [config_path, val_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing saved Notebook 6 result: {path}")
config = pd.read_csv(config_path).iloc[0]
val_pred = pd.read_csv(val_path, parse_dates=["timestamp", "cutoff"])
assert val_pred["row_index"].is_unique
assert len(val_pred) == int(m3_df[SPLIT_COL].eq("validation").sum())
val_rows = m3_df.loc[val_pred["row_index"].astype(int).to_numpy()]
assert val_rows[SPLIT_COL].eq("validation").all(), "Validation row indices shifted when test data was included."
assert (val_rows[TIME_COL].to_numpy() == val_pred["timestamp"].to_numpy()).all(), "Validation timestamp alignment failed."
assert (val_rows[TARGET_COL].astype(int).to_numpy() == val_pred["y_true"].astype(int).to_numpy()).all(), "Validation label alignment failed."
if "amt" not in m3_df.columns:
    raise KeyError("Expected original Sparkov transaction amount column 'amt' in processed feature table")
val_pred["amount"] = pd.to_numeric(val_rows["amt"], errors="raise").to_numpy()
val_pred["date"] = val_pred["timestamp"].dt.floor("D")
print("Locked M3 configuration:", config[["fraud_thres","legit_thres","fraud_reliability","legit_reliability"]].to_dict())
print("Saved validation PR-AUC:", average_precision_score(val_pred.y_true, val_pred.y_score))
print("Validation transactions:", len(val_pred))

Locked M3 configuration: {'fraud_thres': 0.5, 'legit_thres': 0.5, 'fraud_reliability': 0.5, 'legit_reliability': 0.3}
Saved validation PR-AUC: 0.9853224063131583
Validation transactions: 259335


## 2. Operational alert selection (validation only)

Select k by minimising the cost of reviewing the top k scored transactions **per day**, plus the amount of missed fraudulent transactions. `$3.47` per review matches the IEEE-CIS assumed cost; it is a modelling assumption, not a measured Sparkov cost. Candidate k values are filtered against observed Sparkov daily transaction counts; a day with fewer than k transactions simply flags all available transactions.

In [5]:
REVIEW_COST = 3.47
K_VALUES = [5, 10, 15, 20, 25, 50, 75, 100, 150, 200, 300, 400, 500, 750, 1000, 1500, 2000]

def daily_topk_flags(frame, k):
    if k < 0: raise ValueError("k must be nonnegative")
    # Deterministic tie-breaking by row_index.
    ranked = frame.sort_values(["date", "y_score", "row_index"], ascending=[True, False, True])
    selected = ranked.groupby("date", sort=False).cumcount().lt(k)
    return pd.Series(selected.to_numpy(), index=ranked.index).reindex(frame.index).astype(bool)

def operating_metrics(frame, k, review_cost=REVIEW_COST):
    flags = daily_topk_flags(frame, k)
    truth = frame["y_true"].astype(bool)
    tp = int((flags & truth).sum()); fp = int((flags & ~truth).sum())
    fn = int((~flags & truth).sum()); tn = int((~flags & ~truth).sum())
    missed = float(frame.loc[~flags & truth, "amount"].sum())
    review = float(review_cost * (tp + fp))
    return {"k": k, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "alerts": tp + fp, "precision": tp / (tp+fp) if tp+fp else 0.0,
            "recall": tp / (tp+fn) if tp+fn else 0.0,
            "review_cost": review, "missed_fraud_amount": missed,
            "total_cost": review + missed}

k_results = pd.DataFrame([operating_metrics(val_pred, k) for k in K_VALUES])
best_row = k_results.sort_values(["total_cost", "k"]).iloc[0]
BEST_K = int(best_row["k"])
no_alert_cost = float(val_pred.loc[val_pred.y_true.eq(1), "amount"].sum())
print("Validation transactions per day:")
display(val_pred.groupby("date").size().describe())
print(f"Selected k={BEST_K} per day; validation cost={best_row.total_cost:,.2f}; no-alert cost={no_alert_cost:,.2f}")
display(k_results.sort_values("total_cost").head(10))
k_results.to_csv(RESULTS_DIR / "sparkov_validation_k_sensitivity.csv", index=False)
pd.DataFrame([{"selected_k": BEST_K, "review_cost_per_alert": REVIEW_COST,
               "validation_cost": best_row.total_cost}]).to_csv(RESULTS_DIR / "sparkov_selected_alert_policy.csv", index=False)

Validation transactions per day:


count     108.000000
mean     2401.250000
std       764.763504
min      1366.000000
25%      1812.000000
50%      2116.000000
75%      3103.500000
max      4167.000000
dtype: float64

Selected k=50 per day; validation cost=19,817.90; no-alert cost=816,266.27


,k,tp,fp,fn,tn,alerts,precision,recall,review_cost,missed_fraud_amount,total_cost
5,50,1527,3873,11,253924,5400,0.282778,0.992848,18738.0,1079.90,19817.90
6,75,1532,6568,6,251229,8100,0.189136,0.996099,28107.0,94.04,28201.04
7,100,1534,9266,4,248531,10800,0.142037,0.997399,37476.0,63.78,37539.78
4,25,1409,1291,129,256506,2700,0.521852,0.916125,9369.0,40234.59,49603.59
8,150,1535,14665,3,243132,16200,0.094753,0.998049,56214.0,59.72,56273.72
9,200,1536,20064,2,237733,21600,0.071111,0.998700,74952.0,40.51,74992.51
3,20,1314,846,224,256951,2160,0.608333,0.854356,7495.2,82557.74,90052.94
10,300,1536,30864,2,226933,32400,0.047407,0.998700,112428.0,40.51,112468.51
11,400,1536,41664,2,216133,43200,0.035556,0.998700,149904.0,40.51,149944.51
2,15,1153,467,385,257330,1620,0.711728,0.749675,5621.4,157337.56,162958.96


## 3. Generate final chronological test predictions

At every weekly cutoff, a verified-only LightGBM first scores the previous seven days for pseudo-labelling; M3 then trains on verified and weighted pseudo-labelled rows and predicts the upcoming week. **No test-period hyperparameter tuning.**

In [6]:
test_rows = m3_df.loc[m3_df[SPLIT_COL].eq("test")]
if test_rows.empty:
    raise ValueError("No test rows in processed feature table. Check sparkov_split values.")
TEST_START = test_rows[TIME_COL].min()
TEST_END = test_rows[TIME_COL].max().floor("D") + pd.Timedelta(days=1)
print("Test period:", TEST_START, "to", TEST_END)
print("Test transactions:", len(test_rows))

TEST_PRED_PATH = RESULTS_DIR / "sparkov_m3_test_predictions.csv"
if TEST_PRED_PATH.exists():
    test_pred = pd.read_csv(TEST_PRED_PATH, parse_dates=["timestamp", "cutoff"])
    print("Loaded cached test predictions")
else:
    test_pseudo_cache = precompute_pseudo_scores(
        m3_df, TEST_START, TEST_END,
        allowed_splits=("train", "validation", "test"),
    )
    test_pred = run_adaptive_lightgbm(
        frame=m3_df, start=TEST_START, end=TEST_END,
        fraud_thres=float(config["fraud_thres"]),
        legit_thres=float(config["legit_thres"]),
        fraud_reliability=float(config["fraud_reliability"]),
        legit_reliability=float(config["legit_reliability"]),
        pseudo_score_cache=test_pseudo_cache,
        allowed_splits=("train", "validation", "test"),
        scoring_split="test",
    )
    test_pred.to_csv(TEST_PRED_PATH, index=False)

assert len(test_pred) == len(test_rows), "Test predictions must cover every test transaction"
assert test_pred.row_index.is_unique
scored_test = m3_df.loc[test_pred.row_index.astype(int).to_numpy()]
assert scored_test[SPLIT_COL].eq("test").all()
assert (scored_test[TIME_COL].to_numpy() == test_pred.timestamp.to_numpy()).all()
assert (scored_test[TARGET_COL].astype(int).to_numpy() == test_pred.y_true.astype(int).to_numpy()).all()
test_pred["amount"] = pd.to_numeric(scored_test["amt"], errors="raise").to_numpy()
test_pred["date"] = test_pred.timestamp.dt.floor("D")
print("Final M3 test metrics:", summarize_predictions(test_pred))

Test period: 2020-06-21 12:14:25 to 2021-01-01 00:00:00
Test transactions: 555719
Cached 1/28 cutoffs: 2020-06-21 12:14:25 (1,276,707 verified; 19,968 unlabeled candidates)
Cached 2/28 cutoffs: 2020-06-28 12:14:25 (1,296,675 verified; 19,982 unlabeled candidates)
Cached 3/28 cutoffs: 2020-07-05 12:14:25 (1,316,657 verified; 20,025 unlabeled candidates)
Cached 4/28 cutoffs: 2020-07-12 12:14:25 (1,336,682 verified; 20,182 unlabeled candidates)
Cached 5/28 cutoffs: 2020-07-19 12:14:25 (1,356,864 verified; 19,777 unlabeled candidates)
Cached 6/28 cutoffs: 2020-07-26 12:14:25 (1,376,641 verified; 19,898 unlabeled candidates)
Cached 7/28 cutoffs: 2020-08-02 12:14:25 (1,396,539 verified; 19,733 unlabeled candidates)
Cached 8/28 cutoffs: 2020-08-09 12:14:25 (1,416,272 verified; 20,023 unlabeled candidates)
Cached 9/28 cutoffs: 2020-08-16 12:14:25 (1,436,295 verified; 19,637 unlabeled candidates)
Cached 10/28 cutoffs: 2020-08-23 12:14:25 (1,455,932 verified; 19,722 unlabeled candidates)
Cached 

## 4. Static LightGBM comparison

Train **once** using verified labels available at the first test cutoff and score the entire test period. This is a static *verified-only M2 LightGBM* comparator, not a static pseudo-labelling M3 comparator; interpret accordingly.

In [7]:
STATIC_PATH = RESULTS_DIR / "sparkov_static_lightgbm_test_predictions.csv"
if STATIC_PATH.exists():
    static_pred = pd.read_csv(STATIC_PATH, parse_dates=["timestamp"])
else:
    verified_at_start = verified_training_rows(
        m3_df, TEST_START,
        allowed_splits=("train", "validation", "test"),
    )
    X_fit = prepare_features(verified_at_start)
    cats = {col: X_fit[col].cat.categories for col in categorical_cols}
    X_test = prepare_features(test_rows, categories=cats)
    static_model = make_lightgbm(class_weight="balanced")
    static_model.fit(X_fit, verified_at_start[TARGET_COL].astype(int))
    static_pred = pd.DataFrame({
        "row_index": test_rows.index.to_numpy(),
        "timestamp": test_rows[TIME_COL].to_numpy(),
        "y_true": test_rows[TARGET_COL].astype(int).to_numpy(),
        "y_score": static_model.predict_proba(X_test)[:, 1],
    })
    static_pred.to_csv(STATIC_PATH, index=False)
assert set(static_pred.row_index.astype(int)) == set(test_pred.row_index.astype(int))
comparison = pd.DataFrame([
    {"model": "Adaptive M3 LightGBM", **summarize_predictions(test_pred)},
    {"model": "Static verified-only LightGBM", **summarize_predictions(static_pred)},
])
display(comparison)
comparison.to_csv(RESULTS_DIR / "sparkov_final_test_model_comparison.csv", index=False)

,model,transactions,fraud_cases,pr_auc,roc_auc
0,Adaptive M3 LightGBM,555719,2145,0.972629,0.999212
1,Static verified-only LightGBM,555719,2145,0.971653,0.999379


## 5. Locked-policy operational evaluation and weekly performance

Apply **the same BEST_K chosen on validation**, without re-optimising on test. Weekly PR-AUC is descriptive only.

In [8]:
test_operating = operating_metrics(test_pred, BEST_K)
val_operating = operating_metrics(val_pred, BEST_K)
no_alert_test = float(test_pred.loc[test_pred.y_true.eq(1), "amount"].sum())
operating_table = pd.DataFrame([
    {"split": "validation", **val_operating},
    {"split": "test", **test_operating},
])
display(operating_table)
print(f"Test no-alert benchmark: {no_alert_test:,.2f}")
print(f"Test policy cost: {test_operating['total_cost']:,.2f}")
operating_table.to_csv(RESULTS_DIR / "sparkov_final_test_operational_metrics.csv", index=False)

weekly_rows = []
for model_name, pred in [("Adaptive M3", test_pred), ("Static LightGBM", static_pred)]:
    work = pred.copy()
    work["week"] = ((work.timestamp - TEST_START).dt.total_seconds() // (7*86400)).astype(int)
    for week, block in work.groupby("week"):
        weekly_rows.append({"model": model_name, "week": int(week),
                            "transactions": len(block), "fraud_cases": int(block.y_true.sum()),
                            "pr_auc": average_precision_score(block.y_true, block.y_score)
                            if block.y_true.sum() > 0 else np.nan})
weekly = pd.DataFrame(weekly_rows)
display(weekly)
weekly.to_csv(RESULTS_DIR / "sparkov_final_test_weekly_prauc.csv", index=False)

,split,k,tp,fp,fn,tn,alerts,precision,recall,review_cost,missed_fraud_amount,total_cost
0,validation,50,1527,3873,11,253924,5400,0.282778,0.992848,18738.0,1079.90,19817.90
1,test,50,2127,7573,18,546001,9700,0.219278,0.991608,33659.0,525.32,34184.32


Test no-alert benchmark: 1,133,324.68
Test policy cost: 34,184.32


,model,week,transactions,fraud_cases,pr_auc
0,Adaptive M3,0,19982,68,0.960630
1,Adaptive M3,1,20025,100,0.982899
2,Adaptive M3,2,20182,83,0.939308
3,Adaptive M3,3,19777,52,0.954701
4,Adaptive M3,4,19898,99,0.984324
5,Adaptive M3,5,19733,64,0.939024
6,Adaptive M3,6,20023,152,0.976276
7,Adaptive M3,7,19637,82,0.974171
8,Adaptive M3,8,19722,59,0.956596
9,Adaptive M3,9,19605,102,0.985886



## 6. Final Results and Interpretation

### 6.1 Model Selection and Validation

M3 LightGBM was selected as the final Sparkov model based on validation PR-AUC. It achieved a PR-AUC of 0.985322, marginally outperforming M2 (0.985243), while M4 with Isolation Forest anomaly features performed worse (0.983973). The improvement from M2 to M3 was small, suggesting limited incremental benefit from pseudo-labelling on Sparkov.

The selected M3 configuration was fixed before test evaluation:

- Fraud selection threshold: 0.50
- Legitimate selection threshold: 0.50
- Fraud pseudo-label reliability: 0.50
- Legitimate pseudo-label reliability: 0.30

M3 retained weekly retraining, an expanding historical training window, and a seven-day label verification delay.

### 6.2 Operational Threshold Selection

A daily top-k alert policy was adopted instead of a fixed probability cutoff. Because the model is retrained weekly, predicted probability scales may shift between model versions. Ranking transactions within each day provides a consistent investigation workload and reflects a capacity-constrained fraud review process.

Candidate daily alert budgets were evaluated on the validation set using:

**Total Cost = Review Cost × (TP + FP) + Missed Fraud Amount**

A review cost of $3.47 per flagged transaction was assumed, consistent with the IEEE-CIS evaluation framework.

The optimal validation-selected alert budget was **k = 50 transactions per day**, yielding:

| Metric | Validation |
|---|---:|
| Daily alert budget | 50 |
| Total alerts | 5,400 |
| True positives | 1,527 |
| False positives | 3,873 |
| False negatives | 11 |
| Precision | 28.28% |
| Recall | 99.28% |
| Review cost | $18,738.00 |
| Missed fraud amount | $1,079.90 |
| Total operational cost | $19,817.90 |
| No-alert benchmark | $816,266.27 |

Increasing k beyond 50 produced only small improvements in recall while substantially increasing investigation costs. The selected k was therefore fixed before test evaluation.

### 6.3 Final Test-Set Evaluation

The selected M3 configuration was evaluated on the untouched chronological Sparkov test set, comprising 555,719 transactions and 2,145 fraudulent transactions.

The adaptive M3 pipeline retained weekly retraining and the seven-day verification delay. At each cutoff, verified historical labels and eligible weighted pseudo-labels were used for training, without using future labels.

A static verified-only LightGBM model, trained once using labels available at the beginning of the test period, was included as a baseline.

| Metric | Adaptive M3 LightGBM | Static LightGBM |
|---|---:|---:|
| Test PR-AUC | 0.972629 | 0.971653 |
| Test ROC-AUC | 0.999212 | 0.999379 |

Adaptive M3 achieved a marginal PR-AUC improvement of 0.000976 over the static baseline, while the static model achieved a slightly higher ROC-AUC.

M3's PR-AUC declined from 0.985322 on validation to 0.972629 on test. Nevertheless, discrimination remained strong on the unseen test period, despite the lower test fraud prevalence (0.3860% versus 0.5931% on validation).

These results indicate that adaptive M3 generalised well, but its incremental advantage over the static model was limited. The comparison does not isolate the effect of retraining alone because M3 also incorporates pseudo-labelling.

### 6.4 Test-Set Operational Performance

The validation-selected policy of **50 alerts per day** was applied to test predictions without further optimisation.

| Metric | Validation | Test |
|---|---:|---:|
| Daily alert budget | 50 | 50 |
| True positives | 1,527 | 2,127 |
| False positives | 3,873 | 7,573 |
| False negatives | 11 | 18 |
| Total alerts | 5,400 | 9,700 |
| Precision | 28.28% | 21.93% |
| Recall | 99.28% | 99.16% |
| Review cost | $18,738.00 | $33,659.00 |
| Missed fraud amount | $1,079.90 | $525.32 |
| Total operational cost | $19,817.90 | $34,184.32 |
| No-alert benchmark | $816,266.27 | $1,133,324.68 |

On the test set, the selected policy identified 2,127 of 2,145 fraudulent transactions, achieving **99.16% recall** while missing only 18 fraud cases.

The estimated operational cost was $34,184.32, compared with $1,133,324.68 under a no-alert scenario. This represents an approximately **96.98% reduction in estimated cost**, under the assumed investigation-cost and missed-fraud-loss framework.

Test recall remained close to validation recall (99.16% versus 99.28%), suggesting that the validation-selected alert budget transferred effectively to the test period. Precision decreased from 28.28% to 21.93%, consistent in part with the lower fraud prevalence in the test set.

### 6.5 Weekly Performance and Limitations

Weekly PR-AUC was calculated for both models to assess temporal performance.

Across the 27 test weeks containing fraudulent transactions, adaptive M3 achieved higher PR-AUC than the static baseline in 15 weeks, while the static model performed better in 12 weeks. Both models achieved weekly PR-AUC above 0.93 in these evaluable weeks.

Week 27 contained no fraudulent transactions; its PR-AUC was therefore recorded as undefined (NaN) and excluded from weekly PR-AUC comparisons.

Although M3 demonstrated strong discrimination and operational recall, several limitations remain:

1. The incremental PR-AUC improvement over the static baseline was small and cannot be attributed exclusively to retraining.
2. The $3.47 investigation cost and assumption that missed fraud amounts represent losses are modelling assumptions rather than observed operational outcomes.
3. The daily top-k policy assumes a fixed investigation capacity and does not account for variations in case complexity or investigation effectiveness.
4. Sparkov is a synthetic dataset; results may not generalise directly to real-world fraud patterns or operational environments.

### 6.6 Conclusion

The final Sparkov evaluation demonstrated strong out-of-time performance for M3 LightGBM, achieving a test PR-AUC of 0.972629. However, its improvement over static LightGBM was marginal, indicating that the additional complexity of pseudo-labelling and adaptive retraining provided limited incremental predictive benefit on this dataset.

The validation-selected daily top-50 alert policy achieved 99.16% fraud recall on the untouched test set and reduced estimated operational costs by approximately 96.98% relative to a no-alert benchmark.

Overall, the results support the effectiveness of the LightGBM-based fraud ranking system under the experimental assumptions, while highlighting the importance of evaluating adaptive learning methods against strong static baselines.



## Additional M3 Classification Metrics

We evaluate the selected M3 LightGBM model on both validation and test sets using precision and recall at a fixed probability threshold of 0.5, as well as Precision@100 per day.

Precision@100 measures the proportion of fraudulent transactions among the 100 highest-scoring transactions each day. These metrics complement PR-AUC by assessing classification performance at a fixed threshold and under a daily alert budget.


In [1]:

from pathlib import Path
import pandas as pd
from sklearn.metrics import precision_score, recall_score

# Paths relative to the notebooks folder
val_path = Path("../results/m3_rank_aligned_ieee/m3_validation_predictions.csv")
test_path = Path("../results/sparkov_final_test/sparkov_m3_test_predictions.csv")

# Find the test CSV if the output directory differs
if not test_path.exists():
    matches = list(Path("../results").rglob("sparkov_m3_test_predictions.csv"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one saved M3 test prediction CSV, found: {matches}"
        )
    test_path = matches[0]

val_pred = pd.read_csv(val_path, parse_dates=["timestamp"])
test_pred = pd.read_csv(test_path, parse_dates=["timestamp"])

def calculate_metrics(df, split):
    df = df.copy()
    df["date"] = df["timestamp"].dt.floor("D")

    y_true = df["y_true"].astype(int)
    y_score = df["y_score"].astype(float)

    # Precision and recall at threshold 0.5
    y_pred = (y_score >= 0.5).astype(int)

    # Top 100 highest scores per day
    ranked = df.sort_values(
        ["date", "y_score", "row_index"],
        ascending=[True, False, True]
    )
    top100 = ranked.groupby("date").head(100)

    return {
        "Split": split,
        "Precision@0.5": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall@0.5": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "Precision@100/day": top100["y_true"].sum() / len(top100),
    }

metrics = pd.DataFrame([
    calculate_metrics(val_pred, "Validation"),
    calculate_metrics(test_pred, "Test"),
])

display(metrics.round(6))


,Split,Precision@0.5,Recall@0.5,Precision@100/day
0,Validation,0.870220,0.976593,0.142037
1,Test,0.751901,0.967832,0.110155
